# Black-Scholes, Core Greeks, and IV Inversion

## Black-Scholes-Merton Model

1. Assumptions

The Black-Scholes-Merton (BSM) model assumes:

- No arbitrage
- No transaction costs
- Continuous trading
- Constant risk-free rate $r$
- Constant volatility $\sigma$
- Continuous dividend yield $q$
- The stock price follows a geometric Brownian motion (GBM):

$$
dS_t = (\mu - q)S_t\,dt + \sigma S_t\,dW_t,
$$

where:

- $S_t$ is the stock price at time $t$,
- $\mu$ is the expected total return of the stock,
- $q$ is the continuous dividend yield,
- $\sigma$ is the volatility,
- $W_t$ is a standard Brownian motion.

2. Key Idea: Delta Hedging

The key idea of Black-Scholes-Merton is to eliminate the stochastic term $dW_t$
by constructing a locally riskless portfolio.

Consider a portfolio:

- Long one call option with value $V(S,t)$,
- Short $\Delta$ shares of the underlying stock,

where

$$
\Delta = \frac{\partial V}{\partial S}.
$$

Under the no-arbitrage condition, the resulting locally riskless portfolio must earn
the risk-free rate $r$.

This leads to the Black-Scholes-Merton PDE:

$$
\frac{\partial V}{\partial t}
+ \frac{1}{2}\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2}
+ (r-q)S\frac{\partial V}{\partial S}
- rV
= 0.
$$

3. Option Pricing Formula

Let

$$
\tau = T-t
$$

denote the time remaining until maturity.

Define

$$
d_1
=
\frac{
\ln\left(\frac{S}{K}\right)
+
\left(r-q+\frac{1}{2}\sigma^2\right)\tau
}{
\sigma\sqrt{\tau}
},
$$

and

$$
d_2
=
d_1-\sigma\sqrt{\tau}
$$

Then the call price is

$$
C
=
Se^{-q\tau}N(d_1)
-
Ke^{-r\tau}N(d_2),
$$

and the put price is

$$
P
=
Ke^{-r\tau}N(-d_2)
-
Se^{-q\tau}N(-d_1).
$$

where $N(x)$ is the cumulative distribution function (CDF) of the standard normal distribution.

In [66]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

In [67]:
S = 100 # current stock price
K = 100 # strike price
r = 0.05 # continuously compounded risk-free rate
q = 0.02 # continuous dividend yield
T = 0.5 # time to expiry in years
sigma = 0.2 # annualized volatility

In [68]:
class BSM:

    def __init__(self, S, K, r, q, T, sigma):
        self.S = S
        self.K = K
        self.r = r
        self.q = q
        self.T = T
        self.sigma = sigma
        self.d1 = (np.log(S/K) + (r - q + 1/2 * sigma**2) * T) / (sigma * np.sqrt(T))
        self.d2 = self.d1 - sigma * np.sqrt(T)

    def prices(self):
        C = self.S * np.exp(-self.q * self.T) * norm.cdf(self.d1) - self.K * np.exp(-self.r * self.T) * norm.cdf(self.d2)
        P = self.K * np.exp(-self.r * self.T) * norm.cdf(-self.d2) - self.S * np.exp(-self.q * self.T) * norm.cdf(-self.d1)

        return C, P

    def delta_call(self):
        return np.exp(-self.q * self.T) * norm.cdf(self.d1)

    def delta_put(self):
        return -np.exp(-self.q * self.T) * norm.cdf(-self.d1)

    def vega(self):
        return self.S * np.exp(-self.q * self.T) * norm.pdf(self.d1) * np.sqrt(self.T)

    def iv(self, P_market, option_type='call', sigma=0.2, iter=1000, tol=1e-4):

        if option_type=='call':
            idx = 0
        elif option_type=='put':
            idx = 1

        for _ in range(iter):
            trial = BSM(self.S, self.K, self.r,  self.q, self.T, sigma)     

            vega = trial.vega()     
            if vega < 1e-10:
                return sigma, False  

            P_model = trial.prices()[idx]
            sigma_new = sigma - (P_model - P_market) / vega
            if abs(sigma_new - sigma) < tol and abs(P_market - P_model) < tol:
                return sigma_new, True
            
            sigma = sigma_new

        return sigma_new, False

bsm = BSM(S, K, r, q, T, sigma)
C, P = bsm.prices()
print(f"C: {C:.4f}, P: {P:.4f}")

C: 6.3076, P: 4.8336


Validation

1. Put-call parity:

$$
C - P = Se^{-qT} - Ke^{-rT}
$$

2. Bounds:

$$
\max\left(Se^{-qT} - Ke^{-rT}, 0\right) \leq C \leq Se^{-qT}
$$

$$
\max\left(Ke^{-rT} - Se^{-qT}, 0\right) \leq P \leq Ke^{-rT}
$$

In [69]:

parity_lhs = C - P
parity_rhs = S * np.exp(-q * T) - K * np.exp(-r * T)
parity_residual = parity_lhs - parity_rhs
print(f"Parity LHS: {parity_lhs:.4f}, RHS: {parity_rhs:.4f}, Residual: {parity_residual:.4f}. Is the residual close to zero: {abs(parity_residual) < 1e-4}")

call_upper_bound = S * np.exp(-q * T)
call_lower_bound = max(S * np.exp(-q * T) - K * np.exp(-r * T), 0)

put_upper_bound = K * np.exp(-r * T)
put_lower_bound = max(K * np.exp(-r * T) - S * np.exp(-q * T), 0)

print(f"Call bounds: [{call_lower_bound:.4f}, {call_upper_bound:.4f}]. Does the call price fall within the bounds: {C >= call_lower_bound and C <= call_upper_bound}")
print(f"Put bounds: [{put_lower_bound:.4f}, {put_upper_bound:.4f}]. Does the put price fall within the bounds: {P >= put_lower_bound and P <= put_upper_bound}")


Parity LHS: 1.4740, RHS: 1.4740, Residual: 0.0000. Is the residual close to zero: True
Call bounds: [1.4740, 99.0050]. Does the call price fall within the bounds: True
Put bounds: [0.0000, 97.5310]. Does the put price fall within the bounds: True


## Option Greeks

$$
\begin{aligned}
\Delta_C
&= \frac{\partial C}{\partial S}
= e^{-qT}N(d_1) \\[6pt]

\Delta_P
&= \frac{\partial P}{\partial S}
= -e^{-qT}N(-d_1) \\[6pt]

\text{Vega}
&= \frac{\partial V}{\partial \sigma}
= Se^{-qT}\phi(d_1)\sqrt{T}
\end{aligned}
$$

where $\phi(x)$ is the probability density function (PDF) of the standard normal distribution.

In [70]:
delta_call = bsm.delta_call()
delta_put = bsm.delta_put()
vega = bsm.vega()

print(f"Delta call: {delta_call:.4f}, delta put: {delta_put:.4f}, vega: {vega:.4f}")

Delta call: 0.5645, delta put: -0.4256, vega: 27.4958


In [71]:
h = 1e-4

delta_call_fd = (BSM(S+h, K, r, q, T, sigma).prices()[0] - C) / h
vega_fd = (BSM(S, K, r, q, T, sigma+h).prices()[0] - C) / h

print(f"Using finite differences, delta call: {delta_call_fd:.4f}, vega: {vega_fd:.4f}")

Using finite differences, delta call: 0.5645, vega: 27.4958


## Implied Volatility: Newton's Method

1. Newton's Method

Suppose we want to find $x$ such that

$$
f(x)=0.
$$

Start with an initial guess $x_0$.

Using a first-order Taylor approximation of $f(x)$ around $x_0$,

$$
f(x)\approx f(x_0)+f'(x_0)(x-x_0).
$$

Choose $x_1$ such that the approximation is equal to zero:

$$
f(x_0)+f'(x_0)(x_1-x_0)=0.
$$

Solving for $x_1$,

$$
x_1
=
x_0-\frac{f(x_0)}{f'(x_0)}.
$$

More generally, Newton's method gives the iterative update

$$
x_{n+1}
=
x_n-\frac{f(x_n)}{f'(x_n)}
$$

for $n=0,1,2,\ldots$.

Continue iterating until the sequence $(x_n)$ converges.

2. Implied Volatility

Suppose $P$ is the observed market price of an option.

We want to find the implied volatility $\sigma$ such that the Black-Scholes-Merton price matches the market price:

$$
\operatorname{BSM}(\sigma)=P.
$$

Define

$$
f(\sigma)
=
\operatorname{BSM}(\sigma)-P.
$$

Then the implied volatility satisfies

$$
f(\sigma)=0.
$$

Since

$$
f'(\sigma)
=
\frac{\partial \operatorname{BSM}}{\partial \sigma}
=
\operatorname{Vega}(\sigma),
$$

Newton's method gives

$$
\sigma_{n+1}
=
\sigma_n
-
\frac{
\operatorname{BSM}(\sigma_n)-P
}{
\operatorname{Vega}(\sigma_n)
}
$$


In [72]:
estimated_sigma, converged = bsm.iv(P_market=7, option_type='call')

if converged:
    print(f"Implied volatility converged to {estimated_sigma:.4f}")
else:
    print(f"Implied volatility failed to converge")


Implied volatility converged to 0.2252


## Experiment

In [73]:
sigmas = [0.1, 0.2, 0.3, 0.4, 0.5]

for true_sigma in sigmas:
    bsm = BSM(S, K, r, q, T, true_sigma)
    C, P = bsm.prices()
    call_sigma, call_converged = bsm.iv(C, 'call')
    put_sigma, put_converged = bsm.iv(P, 'put')

    print(f"True sigma: {true_sigma:.4f}")
    print(f"Call price: {C:.4f}, put price: {P:.4f}")
    
    if call_converged:
        print(f"Estimated call sigma: {call_sigma:.4f}")
    else:
        print(f"Call sigma failed to converge")

    if put_converged:
        print(f"Estimated put sigma: {put_sigma:.4f}")
    else:
        print(f"Put sigma failed to converge")

    print()

True sigma: 0.1000
Call price: 3.5706, put price: 2.0966
Estimated call sigma: 0.1000
Estimated put sigma: 0.1000

True sigma: 0.2000
Call price: 6.3076, put price: 4.8336
Estimated call sigma: 0.2000
Estimated put sigma: 0.2000

True sigma: 0.3000
Call price: 9.0584, put price: 7.5844
Estimated call sigma: 0.3000
Estimated put sigma: 0.3000

True sigma: 0.4000
Call price: 11.8039, put price: 10.3299
Estimated call sigma: 0.4000
Estimated put sigma: 0.4000

True sigma: 0.5000
Call price: 14.5379, put price: 13.0639
Estimated call sigma: 0.5000
Estimated put sigma: 0.5000



A longer time to maturity gives the stock more time to move, which often increases the option's time value. However, the effect on its price is not always positive.

Deep ITM options tend to have higher prices because they contain more intrinsic value.

In [ ]:
strikes = [60, 80, 100, 120, 140]
times = [7/365, 0.25, 0.5, 0.75, 1]
calls = []
puts = []

for Ti in times:
    for Ki in strikes:
        bsm = BSM(S, Ki, r, q, Ti, sigma)
        C, P = bsm.prices()
        calls.append(C)
        puts.append(P)

calls = pd.DataFrame(np.array(calls).reshape(len(times), len(strikes)), index=times, columns=strikes)
calls = calls.rename_axis(index='T', columns='K')

puts = pd.DataFrame(np.array(puts).reshape(len(times), len(strikes)), index=times, columns=strikes)
puts = puts.rename_axis(index='T', columns='K')

print(f"Stock price: {S}")
print("Call prices:")
print(calls)

print()
print("Put prices:")
print(puts)

Stock price: 100
Call prices:
K               60         80        100           120           140
T                                                                   
0.019178  40.019158  20.038327  1.133159  1.175567e-11  1.005438e-34
0.250000  40.246580  20.526850  4.335886  1.762424e-01  1.550398e-03
0.500000  40.486647  21.216114  6.307635  8.825304e-01  6.498056e-02
0.750000  40.723282  21.985732  7.875256  1.775994e+00  2.751951e-01
1.000000  40.961681  22.764125  9.227006  2.711776e+00  6.195364e-01

Put prices:
K                  60            80        100        120        140
T                                                                   
0.019178  2.316751e-77  9.851026e-17  1.075664  19.923335  39.904167
0.250000  1.510821e-07  3.182568e-02  3.592418  19.184331  38.761195
0.500000  2.581226e-04  2.359238e-01  4.833643  18.914736  37.603385
0.750000  3.753406e-03  5.300919e-01  5.683504  18.848130  36.611220
1.000000  1.557933e-02  8.426121e-01  6.330081  18.839440  3